# 归档材料

保留此材料用于局部机制学习；先修、结论与下游连接需要结合归档索引审查。

[归档索引](../README.md) · [当前学习入口](../../../course/first_loop/README.md)

# 04 · Localization & Mapping：自车到底在哪里？

perception/tracking 处理“别人在哪里”；localization 处理“我在哪里”。本章使用同一个 urban cut-in road frame，模拟 wheel-odometry 漂移、GNSS outage 和 map matching。定位误差会污染所有后续 BEV、tracking、planning 和 closed-loop 指标。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "src" / "ad_tutorial").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import numpy as np
import matplotlib.pyplot as plt

times = np.arange(0.0, 20.0, 0.1)
true_pose = np.c_[0.8 * times, 0.4 * np.sin(times / 3.0)]
rng = np.random.default_rng(33)
odometry = true_pose + np.cumsum(rng.normal(0, [0.015, 0.012], true_pose.shape), axis=0)
gnss = true_pose + rng.normal(0, 0.25, true_pose.shape)
outage = (times >= 8.0) & (times < 13.0)
gnss[outage] = np.nan

In [ ]:
def fuse_pose(odometry_xy, gnss_xy, map_xy=None):
    estimate = odometry_xy.copy()
    for index in range(len(estimate)):
        if np.isfinite(gnss_xy[index]).all():
            estimate[index] = 0.25 * estimate[index] + 0.75 * gnss_xy[index]
    if map_xy is not None:
        estimate = 0.85 * estimate + 0.15 * map_xy
    return estimate

map_centerline = np.c_[0.8 * times, np.zeros_like(times)]
estimate = fuse_pose(odometry, gnss, map_centerline)
error = np.linalg.norm(estimate - true_pose, axis=1)
print({"pose_rmse_m": float(np.sqrt(np.mean(error ** 2))), "outage_seconds": float(outage.sum() * 0.1),
       "max_error_m": float(error.max())})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(true_pose[:, 0], true_pose[:, 1], label="ground truth")
axes[0].plot(odometry[:, 0], odometry[:, 1], alpha=0.7, label="odometry")
axes[0].plot(estimate[:, 0], estimate[:, 1], label="fused + map")
axes[0].set_aspect("equal")
axes[0].legend()
axes[0].set(title="Pose and map alignment", xlabel="x / m", ylabel="y / m")
axes[1].plot(times, error)
axes[1].axvspan(8, 13, color="orange", alpha=0.2, label="GNSS outage")
axes[1].legend()
axes[1].set(title="Localization error over time", xlabel="time / s", ylabel="error / m")
plt.tight_layout()
plt.show()

## 练习：漂移、地图匹配和 ODD

1. 增大 odometry drift，观察 outage 后的恢复时间；
2. 把地图中心线故意平移 1m，说明 map matching 何时会把系统“拉向错误答案”；
3. 把 `max_localization_sigma_m` 写成 Chapter 00 的 ODD gate，并说明 planner 应该减速、冻结，还是退出 ODD。

In [ ]:
drift_scale = np.linspace(0.5, 3.0, 8)
outage_rmse = []
for scale in drift_scale:
    local_odometry = true_pose + np.cumsum(rng.normal(0, [0.015, 0.012], true_pose.shape) * scale, axis=0)
    local = fuse_pose(local_odometry, gnss, map_centerline)
    outage_rmse.append(np.sqrt(np.mean(np.linalg.norm(local[outage] - true_pose[outage], axis=1) ** 2)))
plt.plot(drift_scale, outage_rmse, marker="o")
plt.xlabel("odometry drift multiplier")
plt.ylabel("outage RMSE / m")
plt.title("Localization robustness is an ODD/system concern")
plt.show()

save_numpy_artifact("04_localization.npz", time_s=times, true_pose=true_pose, estimate=estimate, error=error)
save_json_artifact("04_localization_meta.json", {
    "pose_frame": "map",
    "outage": [8.0, 13.0],
    "pose_rmse_m": float(np.sqrt(np.mean(error ** 2))),
    "next": "05_learnable_bev_model.ipynb",
})
print("saved localization artifact")

### 完成标准

你要能区分 pose drift、sensor outage、map error 和 actor tracking error 的观测症状，并把每一种症状映射到一个可测指标和一个系统动作。下一章开始真正训练有 AD 语义的 BEV 模型。